# 04. Multimodal fusion: concat

Nối vector PhoBERT `<s>` và vector pooled CLIP, qua MLP phân loại.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/<team>/IE403.git"   # TODO: sửa thành repo của nhóm
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

## Train

Checkpoint `last.pt` lưu mỗi epoch. Session bị ngắt thì attach output version trước làm input và chạy lại với `--override train.resume_from=<đường dẫn>`.

In [ ]:
!python scripts/train.py --config configs/fusion_concat.yaml

## Kết quả trên test (best checkpoint theo val macro-F1)

Copy dòng kết quả vào `reports/results.md` rồi commit.

In [ ]:
cfg = load_config("configs/fusion_concat.yaml")
out_dir = Path(cfg.paths.output_dir) / cfg.name
metrics = json.loads((out_dir / "test_metrics.json").read_text())
from vimmsd.training.metrics import format_metrics
print(format_metrics(metrics))
display(IPImage(str(out_dir / "confusion_matrix_test.png"), width=450))
print((out_dir / "results_row.md").read_text())

## Phân tích lỗi

In [ ]:
!python scripts/evaluate.py --config configs/fusion_concat.yaml --split test
import pandas as pd
pred = pd.read_csv(out_dir / "test_predictions.csv")
wrong = pred[pred.label != pred.pred]
print(f"{len(wrong)}/{len(pred)} mẫu sai")
wrong.groupby(["label", "pred"]).size().sort_values(ascending=False)

In [ ]:
wrong.sample(min(10, len(wrong)), random_state=0)[["id", "label", "pred", "caption"]]